# Train a bigger GPT on a GPU, then run it in the browser

This notebook trains the **same GPT architecture** as the from-scratch NumPy model in this repo, but in PyTorch on a GPU so it can be much bigger and train far longer. It then exports a `model.json` that is **byte-compatible with `web/gpt.js`**, so the browser demo can serve your better model with no code changes.

## How to use
1. Open this notebook in **Google Colab**.
2. **Runtime -> Change runtime type -> GPU** (a free T4 is plenty).
3. **Runtime -> Run all**. Training takes roughly 10-20 minutes.
4. At the end it downloads `model.json`. Send that file back to me (or drop it in `llm-from-scratch/web/`) and I will wire it into the live demo.

It trains on **tiny Shakespeare** at the character level -- the proven sweet spot for a small GPT, and what produces readable mock-Shakespeare.

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Enable it via Runtime -> Change runtime type -> GPU. It will still run on CPU, just slowly.')

In [ ]:
# Download the training corpus (tiny Shakespeare, ~1.1 MB).
import urllib.request, os
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(URL, 'input.txt')
text = open('input.txt', encoding='utf-8').read()
print('corpus characters:', len(text))
print(text[:250])

In [ ]:
# --- Configuration ------------------------------------------------------
# Bigger = better text but a larger model.json download and slower in-browser
# generation. These defaults (~2.7M params, ~14 MB export) balance quality
# against the browser. Bump n_embd / n_layer / max_iters for more quality.
from dataclasses import dataclass

@dataclass
class Config:
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 192
    block_size: int = 128      # context length (characters)
    dropout: float = 0.2
    batch_size: int = 64
    max_iters: int = 5000
    eval_interval: int = 500
    eval_iters: int = 100
    learning_rate: float = 1e-3
    min_lr: float = 1e-4
    warmup: int = 100
    weight_decay: float = 0.1
    grad_clip: float = 1.0

cfg = Config()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Character-level tokenizer (matches gpt.js type 'char').
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[int(i)] for i in ids)
print('vocab size:', vocab_size)

In [ ]:
# Encode the whole corpus and split into train/val.
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - cfg.block_size - 1, (cfg.batch_size,))
    x = torch.stack([d[i:i + cfg.block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + cfg.block_size] for i in ix])
    return x.to(device), y.to(device)

print('train tokens:', len(train_data), 'val tokens:', len(val_data))

In [ ]:
# --- Model: same architecture as the repo's NumPy GPT --------------------
import math
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.n_head = c.n_head
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        self.rdrop = nn.Dropout(c.dropout)
        self.register_buffer('mask', torch.tril(torch.ones(c.block_size, c.block_size)).view(1, 1, c.block_size, c.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hd))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = self.rdrop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Linear(c.n_embd, 4 * c.n_embd)
        self.proj = nn.Linear(4 * c.n_embd, c.n_embd)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate='tanh')))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c.n_embd)
        self.attn = CausalSelfAttention(c)
        self.ln2 = nn.LayerNorm(c.n_embd)
        self.mlp = MLP(c)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, c, vocab_size):
        super().__init__()
        self.cfg = c
        self.wte = nn.Embedding(vocab_size, c.n_embd)
        self.wpe = nn.Embedding(c.block_size, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for b in self.blocks:
            x = b(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = GPT(cfg, vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')
print(f'estimated model.json download: ~{n_params * 4 * 1.34 / 1e6:.0f} MB')

In [ ]:
# --- Training -----------------------------------------------------------
import time
opt = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, betas=(0.9, 0.95))

def lr_at(it):
    if it < cfg.warmup:
        return cfg.learning_rate * it / cfg.warmup
    if it > cfg.max_iters:
        return cfg.min_lr
    ratio = (it - cfg.warmup) / (cfg.max_iters - cfg.warmup)
    coeff = 0.5 * (1 + math.cos(math.pi * ratio))
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            _, loss = model(*get_batch(split))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

t0 = time.time()
for it in range(cfg.max_iters + 1):
    for g in opt.param_groups:
        g['lr'] = lr_at(it)
    if it % cfg.eval_interval == 0:
        l = estimate_loss()
        print(f'iter {it:5d} | train {l["train"]:.3f} | val {l["val"]:.3f} | {time.time() - t0:.0f}s')
    x, y = get_batch('train')
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    opt.step()
print('training done in', round(time.time() - t0), 's')

In [ ]:
# Quick sample so you can eyeball quality before exporting.
model.eval()
start = torch.tensor([encode('ROMEO:')], dtype=torch.long, device=device)
out = model.generate(start, 400, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))

In [ ]:
# --- Export to model.json (byte-compatible with web/gpt.js) -------------
# gpt.js expects Linear weights as (in, out) row-major; PyTorch stores them
# as (out, in), so each Linear weight is transposed on the way out.
import base64, json
import numpy as np

def b64(t):
    a = t.detach().cpu().numpy().astype('<f4')
    return {'shape': list(a.shape), 'data': base64.b64encode(np.ascontiguousarray(a).tobytes()).decode('ascii')}

w = {}
w['wte.weight'] = b64(model.wte.weight)
w['wpe.weight'] = b64(model.wpe.weight)
for i, b in enumerate(model.blocks):
    p = 'blocks.' + str(i) + '.'
    w[p + 'ln1.gamma'] = b64(b.ln1.weight)
    w[p + 'ln1.beta'] = b64(b.ln1.bias)
    w[p + 'attn.qkv.weight'] = b64(b.attn.qkv.weight.t())
    w[p + 'attn.qkv.bias'] = b64(b.attn.qkv.bias)
    w[p + 'attn.proj.weight'] = b64(b.attn.proj.weight.t())
    w[p + 'attn.proj.bias'] = b64(b.attn.proj.bias)
    w[p + 'ln2.gamma'] = b64(b.ln2.weight)
    w[p + 'ln2.beta'] = b64(b.ln2.bias)
    w[p + 'mlp.fc.weight'] = b64(b.mlp.fc.weight.t())
    w[p + 'mlp.fc.bias'] = b64(b.mlp.fc.bias)
    w[p + 'mlp.proj.weight'] = b64(b.mlp.proj.weight.t())
    w[p + 'mlp.proj.bias'] = b64(b.mlp.proj.bias)
w['ln_f.gamma'] = b64(model.ln_f.weight)
w['ln_f.beta'] = b64(model.ln_f.bias)
w['head.weight'] = b64(model.head.weight.t())

spec = {
    'config': {'vocab_size': vocab_size, 'block_size': cfg.block_size, 'n_layer': cfg.n_layer, 'n_head': cfg.n_head, 'n_embd': cfg.n_embd},
    'tokenizer': {'type': 'char', 'chars': chars},
    'weights': w,
}
with open('model.json', 'w') as f:
    json.dump(spec, f)
print('wrote model.json:', round(os.path.getsize('model.json') / 1e6, 1), 'MB')

In [ ]:
# --- Verify the export matches gpt.js -----------------------------------
# Re-load model.json and run a NumPy mirror of gpt.js's forward pass, then
# compare its logits to the trained PyTorch model. Small diff == the browser
# will produce the same outputs as the model you just trained.
spec2 = json.load(open('model.json'))
W = {k: np.frombuffer(base64.b64decode(v['data']), dtype='<f4').reshape(v['shape']).astype(np.float32) for k, v in spec2['weights'].items()}
C = cfg.n_embd; nh = cfg.n_head; hd = C // nh

def ln(x, g, bb, eps=1e-5):
    mu = x.mean(-1, keepdims=True); xc = x - mu; var = (xc * xc).mean(-1, keepdims=True)
    return xc / np.sqrt(var + eps) * g + bb

def gelu(x):
    c = math.sqrt(2 / math.pi)
    return 0.5 * x * (1 + np.tanh(c * (x + 0.044715 * x ** 3)))

def mirror_logits(ids):
    T = len(ids)
    x = W['wte.weight'][ids] + W['wpe.weight'][np.arange(T)]
    for l in range(cfg.n_layer):
        p = 'blocks.' + str(l) + '.'
        h = ln(x, W[p + 'ln1.gamma'], W[p + 'ln1.beta'])
        qkv = (h @ W[p + 'attn.qkv.weight'] + W[p + 'attn.qkv.bias']).reshape(T, 3, nh, hd)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        out = np.zeros((T, C), dtype=np.float32)
        for head in range(nh):
            att = (q[:, head] @ k[:, head].T) / math.sqrt(hd)
            att = np.where(np.triu(np.ones((T, T)), 1).astype(bool), -1e9, att)
            att = att - att.max(-1, keepdims=True)
            e = np.exp(att); att = e / e.sum(-1, keepdims=True)
            out[:, head * hd:(head + 1) * hd] = att @ v[:, head]
        x = x + (out @ W[p + 'attn.proj.weight'] + W[p + 'attn.proj.bias'])
        h2 = ln(x, W[p + 'ln2.gamma'], W[p + 'ln2.beta'])
        m = gelu(h2 @ W[p + 'mlp.fc.weight'] + W[p + 'mlp.fc.bias'])
        x = x + (m @ W[p + 'mlp.proj.weight'] + W[p + 'mlp.proj.bias'])
    x = ln(x, W['ln_f.gamma'], W['ln_f.beta'])
    return x[-1] @ W['head.weight']

ids = encode(text[:16])
model.eval()
with torch.no_grad():
    torch_logits = model(torch.tensor([ids], device=device))[0][0, -1].cpu().numpy()
js_logits = mirror_logits(ids)
diff = float(np.max(np.abs(torch_logits - js_logits)))
print('max abs logit diff (torch vs gpt.js mirror):', diff)
assert diff < 1e-2, 'export does not match gpt.js -- do not deploy this file'
print('OK: model.json will run identically in the browser.')

In [ ]:
# Download model.json to your computer (Colab only).
try:
    from google.colab import files
    files.download('model.json')
except Exception as e:
    print('Not on Colab or download unavailable; find model.json in the file browser.', e)

## Next steps

You now have a `model.json` trained on a GPU. To put it live on the demo:

- **Easiest:** send me this `model.json` and I will commit it to `llm-from-scratch/web/` and the Pages workflow will serve it directly (no more CPU training in CI).
- **Or yourself:** drop it at `llm-from-scratch/web/model.json` (force-add it, since that path is gitignored: `git add -f llm-from-scratch/web/model.json`), and the deploy will pick it up.

Want even better text? Re-run with a bigger config in the **Configuration** cell (e.g. `n_embd=384, n_layer=6, block_size=256, max_iters=8000`) -- just note the `model.json` download and in-browser speed grow with the model.